# Scrapping the Indian Society for Labour Economics Conference [Website](https://www.isleijle.org/isle-conference-archives/)

> Add blockquote


**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 28.6 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,185 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugi

# Scrapping the Website

In [31]:
# Load HTML from a URL (replace 'your_url' with the actual URL)
url = 'https://www.isleijle.org/isle-conference-archives/'
response = requests.get(url)
html_content = response.content


In [32]:
# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Initialize lists for extracted data
conference_years = np.array([])
volumes = []
issues = []
titles = []
authors = []

# Locate all <h2> tags with class "et_pb_toggle_title" (conference year)
conference_year_tags = soup.find_all('h2', class_='et_pb_toggle_title')


In [33]:
import numpy as np

# Extract data for each conference year
for year_tag in conference_year_tags:
    year = year_tag.text.strip()  # Extract the year
    toggle_content = year_tag.find_next_sibling('div', class_='et_pb_toggle_content clearfix')  # Get sibling div

    if toggle_content:
        # Locate all rows inside the content section
        rows = toggle_content.find_all('div', class_='et_pb_text_inner')
        for row in rows:
            # Extract volume and issue
            volume_issue_header = row.find('h2')
            if volume_issue_header:
                volume_issue = volume_issue_header.text.strip()
                volume, issue = volume_issue.split('::')[1].split('Issue') if 'Issue' in volume_issue else (volume_issue, None)
                volume = volume.strip() if volume else None
                issue = issue.strip() if issue else None

            # Extract individual papers
            papers = row.find_all('p')
            for paper in papers:
                title_tag = paper.find('strong')  # Find the paper title
                author_tag = paper.find('em')  # Find the author

                title = title_tag.text.strip() if title_tag else None
                author = author_tag.text.strip() if author_tag else None

                # Append the data to lists
                # Convert 'Conference Year' to string to handle it properly
                # Convert 'conference_years' to a NumPy array if it's a list
                conference_years = np.array(conference_years)
                # Now you can use .astype()
                conference_years = conference_years.astype(str)
                conference_years = np.append(conference_years, year)
                volumes.append(volume)
                issues.append(issue)
                titles.append(title)
                authors.append(author)

# Create a DataFrame from the extracted data
data = pd.DataFrame({
    'Conference Year': conference_years,
    'Volume': volumes,
    'Issue Number': issues,
    'Paper Title': titles,
    'Author': authors
})


In [6]:
# Display the DataFrame
print(data)

# Save to CSV if needed
data.to_csv('conference_data.csv', index=False)

     Conference Year     Volume Issue Number  \
0               1958   Volume 1          1-2   
1               1958   Volume 1          1-2   
2               1958   Volume 1          1-2   
3               1958   Volume 1          1-2   
4               1958   Volume 1          1-2   
...              ...        ...          ...   
2020            2007  Volume 50            4   
2021            2007  Volume 50            4   
2022            2007  Volume 50            4   
2023            2007  Volume 50            4   
2024            2007  Volume 50            4   

                                            Paper Title  \
0     Wage and Employment Policy of Trade Union in a...   
1                                                  None   
2           Disguised Unemployment or Underemployment ?   
3           Wage Policy and Economic Planning in India,   
4            The Problem of Disparity in Wage , Incomes   
...                                                 ...   
2020  Wage

In [7]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

# Load HTML from a URL (replace 'your_url' with the actual URL)
url = 'https://www.isleijle.org/isle-conference-archives/'
response = requests.get(url)
html_content = response.content

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Initialize lists for extracted data
conference_years = []
volumes = []
issues = []
titles = []
authors = []

# Locate all <h2> tags with class "et_pb_toggle_title" (conference year)
conference_year_tags = soup.find_all('h2', class_='et_pb_toggle_title')

# Extract data for each conference year
for year_tag in conference_year_tags:
    year = year_tag.text.strip()  # Extract the year
    toggle_content = year_tag.find_next_sibling('div', class_='et_pb_toggle_content clearfix')  # Get sibling div

    if toggle_content:
        # Locate all rows inside the content section
        rows = toggle_content.find_all('div', class_='et_pb_text_inner')
        for row in rows:
            # Extract volume and issue
            volume_issue_header = row.find('h2')
            if volume_issue_header:
                volume_issue = volume_issue_header.text.strip()
                volume, issue = volume_issue.split('::')[1].split('Issue') if 'Issue' in volume_issue else (volume_issue, None)
                volume = volume.strip() if volume else None
                issue = issue.strip() if issue else None

            # Extract individual papers
            papers = row.find_all('p')
            for paper in papers:
                title_tag = paper.find('strong')  # Find the paper title
                author_tag = paper.find('em')  # Find the author

                title = title_tag.text.strip() if title_tag else None
                author = author_tag.text.strip() if author_tag else None

                # Clean the title to remove any newline or extra spaces (multi-line titles fix)
                if title:
                    title = " ".join(title.splitlines()).strip()

                # Append the data to lists
                conference_years.append(year)
                volumes.append(volume)
                issues.append(issue)
                titles.append(title)
                authors.append(author)

# Create a DataFrame from the extracted data
data = pd.DataFrame({
    'Conference Year': conference_years,
    'Volume': volumes,
    'Issue Number': issues,
    'Paper Title': titles,
    'Author': authors
})

# Display the DataFrame
print(data)

# Save to CSV if needed
data.to_csv('conference_data2.csv', index=False)


     Conference Year     Volume Issue Number  \
0               1958   Volume 1          1-2   
1               1958   Volume 1          1-2   
2               1958   Volume 1          1-2   
3               1958   Volume 1          1-2   
4               1958   Volume 1          1-2   
...              ...        ...          ...   
2020            2007  Volume 50            4   
2021            2007  Volume 50            4   
2022            2007  Volume 50            4   
2023            2007  Volume 50            4   
2024            2007  Volume 50            4   

                                            Paper Title  \
0     Wage and Employment Policy of Trade Union in a...   
1                                                  None   
2           Disguised Unemployment or Underemployment ?   
3           Wage Policy and Economic Planning in India,   
4            The Problem of Disparity in Wage , Incomes   
...                                                 ...   
2020  Wage

Check out missing values

In [34]:
import pandas as pd

# Load the existing CSV file (replace 'conference_data.csv' with your actual file path)
existing_data = pd.read_csv('conference_data.csv')

# Filter the DataFrame to include only rows where the 'Conference Year' is 2004
data_2004 = existing_data[existing_data['Conference Year'] == 2004]

# Print the filtered data
print(data_2004)


      Conference Year     Volume Issue Number Paper Title Author
1775             2004  Volume 47            1         NaN    NaN
1776             2004  Volume 47            1         NaN    NaN
1777             2004  Volume 47            1         NaN    NaN
1778             2004  Volume 47            1         NaN    NaN
1779             2004  Volume 47            1         NaN    NaN
...               ...        ...          ...         ...    ...
1832             2004  Volume 47            4         NaN    NaN
1833             2004  Volume 47            4         NaN    NaN
1834             2004  Volume 47            4         NaN    NaN
1835             2004  Volume 47            4         NaN    NaN
1836             2004  Volume 47            4         NaN    NaN

[62 rows x 5 columns]


Save to drive

In [37]:
# Save the updated DataFrame back to CSV
existing_data.to_csv('updated_conference_data.csv', index=False)

Manual Work to be done to correct for these cases:



1.   Empty cells in the Author Column
2.   Empty cells in the Paper Title Column



Count the authors

Note: Load the files after dealing with missing values manually

In [38]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('/content/drive/My Drive/ISLE.csv')

# Function to clean and count authors
def clean_and_count_authors(row):
    # If 'Author' is NaN or empty, return 0 authors
    if pd.isna(row['Author']) or row['Author'].strip() == "":
        return 0

    # Remove any leading commas
    author_names = re.sub(r'^\s*,\s*', '', row['Author'])

    # Split by comma to count the number of authors
    authors = author_names.split(',')

    # Update 'Number of Authors' based on the comma-separated names
    return len(authors)

# Apply the cleaning function to the 'Author' column and update 'Number of Authors'
df['Number of Authors'] = df.apply(clean_and_count_authors, axis=1)

# Drop the 'Co-Authors' column
df.drop(columns=['Co-Authors'], inplace=True)

# Save the resulting dataset
df.to_csv('/content/drive/My Drive/ISLE_clean.csv', index=False)

# If you want to view the updated dataset
# print(df.head())


In [39]:
print(df.head())

   Conference Year    Volume Issue Number  \
0             1958  Volume 1          1-2   
1             1958  Volume 1          1-2   
2             1958  Volume 1          1-2   
3             1958  Volume 1          1-2   
4             1958  Volume 1          1-2   

                                         Paper Title            Author  \
0  Wage and Employment Policy of Trade Union in a...      Banerjee P C   
1            Housing the Industrial Worker in India,        Saxena S P   
2        Disguised Unemployment or Underemployment ?         Singh V B   
3        Wage Policy and Economic Planning in India,  Srivastava H G P   
4         The Problem of Disparity in Wage , Incomes     Upadhyaya P K   

   Number of Authors  
0                  1  
1                  1  
2                  1  
3                  1  
4                  1  
